In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import tqdm
import sys
import os

In [3]:
from sentence_transformers import SentenceTransformer

encoding_model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
import dotenv
import os

config = dotenv.dotenv_values(".env")
replace_keys = ["JAVA_HOME", "FUSEKI_HOME"]
append_keys = ["PATH"]
for key, value in config.items():
    # append to os.environ
    if key in append_keys:
        os.environ[key] = f"{os.environ.get(key, '')}:{value}"
    elif key in replace_keys:
        os.environ[key] = value

# check and compare values in fuseki log
os.environ["OPENBLAS_NUM_THREADS"] = "4"

In [5]:
sys.path.append(".")

In [6]:
from utils.datasets import SimpleSet, BerlinSparqlBenchmark
from utils.dbs.qlever import QleverDB
from utils.dbs.fuseki import FusekiDB
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.fuseki_native import FusekiDBNative
from utils.datasets.base_dataset import QUERY_DIFFICULTY, QUERY_TYPE
from pathlib import Path
import pandas as pd
import numpy as np
from utils.datasets.base_dataset import DataTensor

2026-08-26 15:56:29,689 - INFO - Loading faiss with AVX512 support.
2026-08-26 15:56:29,691 - INFO - Could not load library with AVX512 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx512'")
2026-08-26 15:56:29,692 - INFO - Loading faiss with AVX2 support.
2026-08-26 15:56:29,692 - INFO - Could not load library with AVX2 support due to:
ModuleNotFoundError("No module named 'faiss.swigfaiss_avx2'")
2026-08-26 15:56:29,692 - INFO - Loading faiss.
2026-08-26 15:56:29,754 - INFO - Successfully loaded faiss.


In [7]:
powers = np.arange(0, 6)  # extend on a more powerful machine
sizes = 10**powers

In [8]:
datasets: dict[int, BerlinSparqlBenchmark] = {}
raw_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Running BDSDM generation for size {size}...")
    dataset = BerlinSparqlBenchmark(base_dir=Path(f"./data/bsbm_{power}"), n=size)
    dataset.setup()
    datasets[power] = dataset
    # raw_sizes[power] = dataset.get_triple_count()

2026-08-26 15:56:29,893 - INFO - BSBM dataset already exists in data/bsbm_0, skipping generation
2026-08-26 15:56:29,895 - INFO - BSBM dataset already exists in data/bsbm_1, skipping generation
2026-08-26 15:56:29,902 - INFO - BSBM dataset already exists in data/bsbm_2, skipping generation
2026-08-26 15:56:29,902 - INFO - BSBM dataset already exists in data/bsbm_3, skipping generation
2026-08-26 15:56:29,903 - INFO - BSBM dataset already exists in data/bsbm_4, skipping generation
2026-08-26 15:56:29,903 - INFO - BSBM dataset already exists in data/bsbm_5, skipping generation


Running BDSDM generation for size 1...
Running BDSDM generation for size 10...
Running BDSDM generation for size 100...
Running BDSDM generation for size 1000...
Running BDSDM generation for size 10000...
Running BDSDM generation for size 100000...


In [9]:
encoded_sizes: dict[int, int] = {}
for power, size in zip(powers, sizes):
    print(f"Encoding dataset of size {size}/power {power}...")
    dataset = datasets[power]
    encoded_sizes[power] = dataset.encode(encoding_model)
    #  dataset.get_triple_count(encoded=True)

Encoding dataset of size 1/power 0...
Encoded TTL file already exists at data/bsbm_0/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10/power 1...
Encoded TTL file already exists at data/bsbm_1/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100/power 2...
Encoded TTL file already exists at data/bsbm_2/dataset_encoded.nt, skipping encoding
Encoding dataset of size 1000/power 3...
Encoded TTL file already exists at data/bsbm_3/dataset_encoded.nt, skipping encoding
Encoding dataset of size 10000/power 4...
Encoded TTL file already exists at data/bsbm_4/dataset_encoded.nt, skipping encoding
Encoding dataset of size 100000/power 5...
Encoded TTL file already exists at data/bsbm_5/dataset_encoded.nt, skipping encoding


## BSBM queries

### Simple Use case: 
find 10 products with a specific encoded label


### Complex Use case: 
For a specific product find 10 other similar products via their product label. 


In [10]:
test_label = "house furniture storage container"
test_tensor = DataTensor.from_numpy(encoding_model.encode(test_label))

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
test_tensor.to_literal().n3()

'"{\\"data\\": [0.0019237988162785769, 0.0590696781873703, -0.06259278953075409, -0.008788960985839367, 0.08017757534980774, 0.0004897424369119108, 0.04901154339313507, -0.016156313940882683, -0.045378170907497406, 0.011929638683795929, -0.003551348578184843, -0.003478354075923562, 0.02326440066099167, 0.024490000680088997, -0.023595565930008888, -0.07382390648126602, 0.014226831495761871, -0.00022307882318273187, -0.031117543578147888, 0.07163398712873459, -0.04559384658932686, 0.04442799091339111, 0.00043878634460270405, -0.0040442910976707935, 0.051731135696172714, 0.08436278998851776, -0.0286672692745924, -0.032494235783815384, 0.058949705213308334, -0.005107350647449493, 0.09506019204854965, -0.029170077294111252, -0.07172349095344543, 0.03642681986093521, 0.017889831215143204, 0.07774496078491211, -0.010582796297967434, -0.027199435979127884, -0.014839786104857922, 0.0049964687786996365, -0.051953013986349106, -0.01563340798020363, -0.0004883426008746028, 0.012572056613862514, -0

In [12]:
base_bsbm_set = datasets[4]
db = QleverDBNative(
    id="test",
    base_dir=Path(f"./scratch/bsbm/{base_bsbm_set.base_dir.name}"),
    dataset=base_bsbm_set,
    use_encoded_ttl=True,
)
possible_queries = db.get_queries(test_tensor)
print(possible_queries)


2026-08-26 15:56:30,637 - WARNING - Killing any existing process using port 26043 before starting the server
2026-08-26 15:56:30,658 - ERROR - Command failed with return code 1
2026-08-26 15:56:30,660 - INFO - Initialized QLeverDBNative with id=test, port_id=26043, dataset=bsbm, name=QLever Native (Extended), use_encoded_ttl=True, endpoint=http://localhost:26043/test-with-tidx/sparql


{<QUERY_DIFFICULTY.EASY: 'easy'>: {<QUERY_TYPE.EMBEDDED: 'embedded'>: '\nPREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>\nPREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>\nPREFIX dtf: <https://w3id.org/rdf-tensor/functions#>\nPREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>\nPREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>\nSELECT DISTINCT ?product  ?vector ?dist \nWHERE {\n?product rdf:label_embedding ?vector .\n?product bsbmv:productFeature ?feat .\nBIND(dtf:cosineSimilarity(?vector, "{\\"data\\": [0.0019237988162785769, 0.0590696781873703, -0.06259278953075409, -0.008788960985839367, 0.08017757534980774, 0.0004897424369119108, 0.04901154339313507, -0.016156313940882683, -0.045378170907497406, 0.011929638683795929, -0.003551348578184843, -0.003478354075923562, 0.02326440066099167, 0.024490000680088997, -0.023595565930008888, -0.07382390648126602, 0.014226831495761871, -0.00022307882318273187, -0.031117543578147888, 0.071633987

In [13]:
from typing import Literal


def get_query_index_variations(
    embedding: DataTensor,
    numNN: int = 1,
    searchK: int = 16,
    mode: Literal["naive", "hsnw", "ivf"] = "naive",
) -> str:
    args = f"""
    _:config tensorIndex:numNN {numNN} ;
    tensorIndex:left ?query_vector ;
    tensorIndex:bindDistance ?dist ;
    tensorIndex:payload ?product ;
    tensorIndex:algorithm tensorIndex:{mode} ;
    tensorIndex:distance tensorIndex:dot ;
    """
    if mode != "naive":
        args += f"""
        tensorIndex:experimentalRightCacheName "easy_index_{mode}_{searchK}" ;
        tensorIndex:searchK {searchK} ;
        """
    if mode == "ivf":
        args += """
        tensorIndex:kIVF 16 ;
        """
    return f"""
PREFIX dt: <https://w3id.org/rdf-tensor/datatypes#>
PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
PREFIX bsbmi: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/instances/>
PREFIX bsbmv: <http://www4.wiwiss.fu-berlin.de/bizer/bsbm/v01/vocabulary/>
PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
SELECT DISTINCT ?product  ?vector ?dist
WHERE {{
SERVICE tensorIndex: {{
        {args}
    tensorIndex:right ?vector .
       {{
            {{
                SELECT DISTINCT ?product ?vector WHERE {{
                    ?product rdf:label_embedding ?vector .
                    ?product bsbmv:productFeature ?feat .
                }} GROUP BY ?product ?vector
            }}
        }}
    }}
    VALUES (?query_vector) {{ ({embedding.to_literal().n3()}) }}
}} ORDER BY DESC(?dist)
LIMIT 16
"""


In [14]:
# 1. get reference results using naive
k = 16
with db:
    query = get_query_index_variations(test_tensor, numNN=k, mode="naive")
    print("Running naive query...")
    results_naive = db.query(query)
results_naive

2026-08-26 15:56:30,850 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_4/db/test-with-tidx, base_dir=scratch/bsbm/bsbm_4
2026-08-26 15:56:30,853 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-08-26 15:56:30,853 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-08-26 15:56:30,855 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-08-26 15:56:30,856 - INFO - Stopping server!
2026-08-26 15:56:30,873 - ERROR - Command failed with return code 1
2026-08-26 15:56:30,874 - INFO - Starting QLever server on port 26043
2026-08-26 15:56:30,875 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 15:56:30,891 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-08-26 15:56:31,914 - INFO - Server is up and 

Running naive query...
 3195182

26043/tcp:          


,product,vector,dist
0,bsbmi:dataFromProducer42/Product1985,"{""data"": [-0.0539478175342083, 0.0999507904052...",0.4656711220741
1,bsbmi:dataFromProducer204/Product9871,"{""data"": [-0.01911473087966442, 0.044213950634...",0.4265948832035
2,bsbmi:dataFromProducer126/Product6035,"{""data"": [-0.02710348181426525, 0.094371080398...",0.4230341315269
3,bsbmi:dataFromProducer90/Product4341,"{""data"": [-0.04623434692621231, 0.029358392581...",0.4177694022655
4,bsbmi:dataFromProducer107/Product5062,"{""data"": [-0.03348814323544502, 0.051626317203...",0.4087206721306
5,bsbmi:dataFromProducer58/Product2673,"{""data"": [-0.019933708012104034, 0.01705851219...",0.4059612154961
6,bsbmi:dataFromProducer103/Product4854,"{""data"": [0.03547070547938347, 0.0046439440920...",0.3998066782951
7,bsbmi:dataFromProducer77/Product3653,"{""data"": [-0.05063876882195473, 0.055860631167...",0.3955110013485
8,bsbmi:dataFromProducer189/Product9168,"{""data"": [-0.02634534426033497, 0.026425562798...",0.3915752768517
9,bsbmi:dataFromProducer37/Product1658,"{""data"": [-0.10466379672288895, 0.054606143385...",0.3833258748055


In [15]:
# 2. get results using ivf
with db:
    query = get_query_index_variations(test_tensor, numNN=k, mode="ivf", searchK=k)
    print("Running ivf query...")
    results_idx = db.query(query)
results_idx

2026-08-26 15:56:32,266 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_4/db/test-with-tidx, base_dir=scratch/bsbm/bsbm_4
2026-08-26 15:56:32,267 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-08-26 15:56:32,267 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-08-26 15:56:32,268 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-08-26 15:56:32,268 - INFO - Stopping server!


2026-08-26 15:56:32,449 - ERROR - Command failed with return code 1
2026-08-26 15:56:32,450 - INFO - Starting QLever server on port 26043
2026-08-26 15:56:32,451 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 15:56:32,455 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-08-26 15:56:33,474 - INFO - Server is up and responding to queries


Running ivf query...


2026-08-26 15:56:33,734 - INFO - Stopping server!


 3195273

26043/tcp:          


,product,vector,dist
0,bsbmi:dataFromProducer42/Product1985,"{""data"": [-0.0539478175342083, 0.0999507904052...",0.4656710624695
1,bsbmi:dataFromProducer204/Product9871,"{""data"": [-0.01911473087966442, 0.044213950634...",0.4265949130058
2,bsbmi:dataFromProducer126/Product6035,"{""data"": [-0.02710348181426525, 0.094371080398...",0.4230340719223
3,bsbmi:dataFromProducer90/Product4341,"{""data"": [-0.04623434692621231, 0.029358392581...",0.4177693724632
4,bsbmi:dataFromProducer107/Product5062,"{""data"": [-0.03348814323544502, 0.051626317203...",0.4087206721306
5,bsbmi:dataFromProducer58/Product2673,"{""data"": [-0.019933708012104034, 0.01705851219...",0.4059612154961
6,bsbmi:dataFromProducer103/Product4854,"{""data"": [0.03547070547938347, 0.0046439440920...",0.3998066186905
7,bsbmi:dataFromProducer77/Product3653,"{""data"": [-0.05063876882195473, 0.055860631167...",0.3955109715462
8,bsbmi:dataFromProducer189/Product9168,"{""data"": [-0.02634534426033497, 0.026425562798...",0.391575217247
9,bsbmi:dataFromProducer37/Product1658,"{""data"": [-0.10466379672288895, 0.054606143385...",0.3833258748055


In [16]:
import time

from utils.helpers import recall_at_k, precision_at_k, ndcgscore_query

In [17]:
recall_at_k(results_idx, results_naive, k=16)

1.0

In [18]:
out_dir = Path("./scratch") / "results" / "bsbm" / "index_tests"
out_dir.mkdir(parents=True, exist_ok=True)

In [19]:


with db:
    runs = 256
    search_ks = {
        "hnsw": np.arange(0, 33, 4).tolist(),
        "ivf": np.arange(0, 17, 2).tolist(),
    }
    for m, search_k_list in search_ks.items():
        search_ks[m] = [1] + search_k_list[1:]

    modes = ["hnsw", "ivf"]
    total = runs * len(modes) * sum(len(s) for s in search_ks.values())
    g = tqdm.tqdm(total=total, desc="Running index queries")

    def run_index_query(
        test_tensor: DataTensor, mode: Literal["naive", "hnsw", "ivf"], search_k: int
    ):
        query = get_query_index_variations(
            test_tensor, numNN=k, searchK=search_k, mode=mode
        )
        return db.raw_query(query)

    timing_results = []
    for mode in modes:
        for search_k in search_ks[mode]:
            run_index_query(test_tensor, mode, search_k)  # warmup
            for run in range(runs):
                noised_tensor = DataTensor.from_numpy(
                    test_tensor.data + np.random.normal(scale=1, size=test_tensor.shape)
                )
                reference_results = db.q_to_df_values(
                    run_index_query(noised_tensor, "naive", search_k)
                )
                g.update(1)
                g.set_description(
                    f"Running {mode} query, run {run + 1}/{runs}, searchK={search_k}"
                )
                result = {}
                start = time.time()
                resp = run_index_query(noised_tensor, mode, search_k)
                end = time.time()
                results_df = db.q_to_df_values(resp)

                result["mode"] = mode
                result["run"] = run
                result["time"] = end - start
                result["searchK"] = search_k
                result["ndcg_score"] = ndcgscore_query(
                    results_df, reference_results, k=k
                )
                result["recall_score"] = recall_at_k(results_df, reference_results, k=k)
                timing_results.append(result)
    g.close()

2026-08-26 15:56:34,155 - INFO - Setting up QLeverDBNative db_dir=scratch/bsbm/bsbm_4/db/test-with-tidx, base_dir=scratch/bsbm/bsbm_4
2026-08-26 15:56:34,157 - INFO - Logging QLever setup to scratch/bsbm/bsbm_4/db/test-with-tidx/test-with-tidx_run.log
2026-08-26 15:56:34,158 - INFO - Loading dataset into QLever server from data/bsbm_4/dataset.nt
2026-08-26 15:56:34,158 - WARNING - DB directory scratch/bsbm/bsbm_4/db/test-with-tidx already exists!
2026-08-26 15:56:34,158 - INFO - Stopping server!
2026-08-26 15:56:34,349 - ERROR - Command failed with return code 1
2026-08-26 15:56:34,351 - INFO - Starting QLever server on port 26043
2026-08-26 15:56:34,352 - INFO - Running command: qlever-server -i test-with-tidx --port 26043 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-08-26 15:56:34,355 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26043/test-with-tidx/sparql)
2026-08-26 15:56:35,377 - INFO - Server is up and 

 3195409

26043/tcp:          


In [20]:
timing_results_df = pd.DataFrame(timing_results)
timing_results_df.to_csv(out_dir / "index_query_timings.csv", index=False)


In [21]:
noised_tensor

DataTensor(data=[-1.0934408979566486, -0.8316234281405617, 0.7200187520612369, 1.657393414002669, 1.1689421158282713, -1.9292894210454503, -1.8869297960954907, 0.6519762874802231, -0.6435978267167665, 2.2866904413101476, 0.5969567001802718, -0.6376036542095411, 0.14153797736628138, 0.298561568213432, -0.7964478663246412, -0.07325718188311772, -1.7582347197012231, 0.9615120954804672, -0.3928620455593868, -0.050543164482468345, 0.0612918147903225, 1.1690490132031188, -0.6519207448672294, -0.001276216817944513, -0.32532717182382187, 0.9790770697184278, 0.4516095476787757, 0.29790988231912546, -0.17636417209809432, -1.4878941138443689, -0.19196605348568407, -1.1069437919759297, -0.39265182725697667, 0.20079427767868377, 1.1138986511594304, 1.7181946098671212, -2.1340395265975105, 0.36096469908894885, 0.01746441832037062, 0.27480651153508273, 2.0610008336906502, -0.5845523122307389, -0.8727111759324576, -0.4179327718230847, 0.5962259159215634, 1.7744646448202943, -0.6807041990924115, -0.147